In [1]:
!pip install pandas requests matplotlib

In [17]:
import requests
import pandas as pd
import numpy as np
import pytz
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
from xgboost import XGBClassifier, XGBRegressor
from datetime import datetime, timedelta


# API DETAILS

API_KEY = "212e6e4959b9694425e3ac449a567829"
BASE_URL = "https://api.openweathermap.org/data/2.5/"


# GET CURRENT WEATHER

city = input("Enter the city name: ")

url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
response = requests.get(url)
data = response.json()

current_weather = {
    "current_temp": data["main"]["temp"],
    "feels_like": data["main"]["feels_like"],
    "temp_min": data["main"]["temp_min"],
    "temp_max": data["main"]["temp_max"],
    "humidity": data["main"]["humidity"],
    "pressure": data["main"]["pressure"],
    "description": data["weather"][0]["description"],
    "country": data["sys"]["country"],
    "wind_gust_speed": data["wind"]["speed"],
    "wind_gust_dir": data["wind"]["deg"]
}

print(f"\nFetched current weather for {city}, {current_weather['country']}")


# READ HISTORICAL DATA

df = pd.read_csv("weather.csv")
df = df.dropna()
df = df.drop_duplicates()

print(f"Historical data loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ENCODE CATEGORICAL COLUMNS


le_wind = LabelEncoder()
le_rain = LabelEncoder()

df["WindGustDir"] = le_wind.fit_transform(df["WindGustDir"])
df["RainTomorrow"] = le_rain.fit_transform(df["RainTomorrow"])

# FEATURES AND TARGET

X = df[["MinTemp", "MaxTemp", "WindGustDir", "WindGustSpeed", "Humidity", "Pressure", "Temp"]]
y = df["RainTomorrow"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# SCALE FEATURES (for SVM, KNN, Logistic Regression)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# DEFINE ALL CLASSIFIERS

classifiers = {
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost":              XGBClassifier(n_estimators=100, random_state=42, verbosity=0),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
    "SVM":                  SVC(probability=True, random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
    "Decision Tree":        DecisionTreeClassifier(random_state=42),
}


# TRAIN AND EVALUATE ALL CLASSIFIERS

print("\n========== RAIN PREDICTION MODEL COMPARISON ==========\n")

accuracy_scores = {}
best_model_name = None
best_model = None
best_accuracy = 0

for name, clf in classifiers.items():

    # SVM, KNN, Logistic Regression work better with scaled data
    if name in ["SVM", "KNN", "Logistic Regression"]:
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    accuracy_scores[name] = acc

    print(f"--- {name} ---")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=["No Rain", "Rain"]))

    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_model = clf


# ACCURACY SUMMARY TABLE

print("\n========== ACCURACY SUMMARY ==========\n")
print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)

for name, acc in sorted(accuracy_scores.items(), key=lambda x: x[1], reverse=True): 
    marker = " <- Best Model" if name == best_model_name else ""

print(f"{name:<25} {acc * 100:>9.2f}%{marker}")

print(f"\nBest Classifier: {best_model_name} with {best_accuracy * 100:.2f}% accuracy")

# VOTING ENSEMBLE CLASSIFIER

print("\n========== VOTING ENSEMBLE CLASSIFIER ==========\n")

voting_clf = VotingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=100, random_state=42)),
        ("xgb", XGBClassifier(n_estimators=100, random_state=42, verbosity=0)),
        ("gb",  GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ],
    voting="soft"
)

voting_clf.fit(X_train, y_train)
y_pred_voting = voting_clf.predict(X_test)
voting_acc = accuracy_score(y_test, y_pred_voting)

print(f"Voting Ensemble Accuracy: {voting_acc * 100:.2f}%")
print(classification_report(y_test, y_pred_voting, target_names=["No Rain", "Rain"]))

if voting_acc > best_accuracy:
    best_accuracy = voting_acc
    best_model_name = "Voting Ensemble"
    best_model = voting_clf
    print(f"Voting Ensemble beats all individual models!")


# WIND DIRECTION ENCODING

wind_deg = current_weather["wind_gust_dir"] % 360

compass_points = [
    ("N",  0,     22.5),
    ("NE", 22.5,  67.5),
    ("E",  67.5,  112.5),
    ("SE", 112.5, 157.5),
    ("S",  157.5, 202.5),
    ("SW", 202.5, 247.5),
    ("W",  247.5, 292.5),
    ("NW", 292.5, 337.5),
    ("N",  337.5, 360)
]

compass_direction = "N"

for point, start, end in compass_points:
    if start <= wind_deg < end:
        compass_direction = point
        break

if compass_direction in le_wind.classes_:
    compass_direction_encoded = le_wind.transform([compass_direction])[0]
else:
    compass_direction_encoded = 0


# CURRENT DATA FOR PREDICTION

current_data = {
    "MinTemp":      current_weather["temp_min"],
    "MaxTemp":      current_weather["temp_max"],
    "WindGustDir":  compass_direction_encoded,
    "WindGustSpeed": current_weather["wind_gust_speed"],
    "Humidity":     current_weather["humidity"],
    "Pressure":     current_weather["pressure"],
    "Temp":         current_weather["current_temp"]
}

current_df = pd.DataFrame([current_data])
current_df_scaled = scaler.transform(current_df)


# RAIN PREDICTIONS FROM ALL MODELS

print("\n========== RAIN PREDICTION FOR TODAY ==========\n")

rain_results = {}

for name, clf in classifiers.items():
    if name in ["SVM", "KNN", "Logistic Regression"]:
        pred = clf.predict(current_df_scaled)[0]
    else:
        pred = clf.predict(current_df)[0]

    rain_results[name] = "Yes" if pred else "No"
    print(f"{name:<25}: Rain → {rain_results[name]}")

voting_pred = voting_clf.predict(current_df)[0]
rain_results["Voting Ensemble"] = "Yes" if voting_pred else "No"
print(f"{'Voting Ensemble':<25}: Rain → {rain_results['Voting Ensemble']}")

# Final rain prediction using best model
if best_model_name in ["SVM", "KNN", "Logistic Regression"]:
    final_rain_prediction = best_model.predict(current_df_scaled)[0]
else:
    final_rain_prediction = best_model.predict(current_df)[0]

print(f"\nFinal Rain Prediction (by best model — {best_model_name}): {'Yes' if final_rain_prediction else 'No'}")


# REGRESSION: TEMPERATURE FUTURE PREDICTION

print("\n========== TEMPERATURE FORECASTING MODEL COMPARISON ==========\n")

X_temp = np.array(df["Temp"].iloc[:-1]).reshape(-1, 1)
y_temp = np.array(df["Temp"].iloc[1:])

X_temp_train, X_temp_test, y_temp_train, y_temp_test = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)

regressors = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor":       XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    "Linear Regression":       LinearRegression(),
}

temp_mse_scores = {}
best_temp_model_name = None
best_temp_model = None
best_temp_mse = float("inf")

for name, reg in regressors.items():
    reg.fit(X_temp_train, y_temp_train)
    y_temp_pred = reg.predict(X_temp_test)
    mse = mean_squared_error(y_temp_test, y_temp_pred)
    rmse = np.sqrt(mse)
    temp_mse_scores[name] = rmse
    print(f"{name:<30} RMSE: {rmse:.4f}")

    if mse < best_temp_mse:
        best_temp_mse = mse
        best_temp_model_name = name
        best_temp_model = reg

print(f"\nBest Temperature Regressor: {best_temp_model_name} (RMSE: {np.sqrt(best_temp_mse):.4f})")


# REGRESSION: HUMIDITY FUTURE PREDICTION

print("\n========== HUMIDITY FORECASTING MODEL COMPARISON ==========\n")

X_hum = np.array(df["Humidity"].iloc[:-1]).reshape(-1, 1)
y_hum = np.array(df["Humidity"].iloc[1:])

X_hum_train, X_hum_test, y_hum_train, y_hum_test = train_test_split(
    X_hum, y_hum, test_size=0.2, random_state=42
)

hum_regressors = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor":       XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    "Linear Regression":       LinearRegression(),
}

hum_mse_scores = {}
best_hum_model_name = None
best_hum_model = None
best_hum_mse = float("inf")

for name, reg in hum_regressors.items():
    reg.fit(X_hum_train, y_hum_train)
    y_hum_pred = reg.predict(X_hum_test)
    mse = mean_squared_error(y_hum_test, y_hum_pred)
    rmse = np.sqrt(mse)
    hum_mse_scores[name] = rmse
    print(f"{name:<30} RMSE: {rmse:.4f}")

    if mse < best_hum_mse:
        best_hum_mse = mse
        best_hum_model_name = name
        best_hum_model = reg

print(f"\nBest Humidity Regressor: {best_hum_model_name} (RMSE: {np.sqrt(best_hum_mse):.4f})")


# FUTURE TEMPERATURE PREDICTIONS (next 5 hours)

future_temp = []
temp_val = current_weather["current_temp"]

for i in range(5):
    next_temp = best_temp_model.predict(np.array([[temp_val]]))[0]
    future_temp.append(next_temp)
    temp_val = next_temp


# FUTURE HUMIDITY PREDICTIONS (next 5 hours)

future_humidity = []
hum_val = current_weather["humidity"]

for i in range(5):
    next_hum = best_hum_model.predict(np.array([[hum_val]]))[0]
    future_humidity.append(next_hum)
    hum_val = next_hum


# FUTURE TIMES

timezone = pytz.timezone("Asia/Kolkata")
now = datetime.now(timezone)

future_times = []

for i in range(1, 6):
    future_time = now + timedelta(hours=i)
    future_times.append(future_time.strftime("%I:%M %p"))


# FINAL WEATHER REPORT

print("         FINAL WEATHER REPORT           ")

print(f"City              : {city}, {current_weather['country']}")
print(f"Current Temp      : {current_weather['current_temp']} °C")
print(f"Feels Like        : {current_weather['feels_like']} °C")
print(f"Min Temperature   : {current_weather['temp_min']} °C")
print(f"Max Temperature   : {current_weather['temp_max']} °C")
print(f"Humidity          : {current_weather['humidity']}%")
print(f"Weather           : {current_weather['description']}")
print(f"Wind Speed        : {current_weather['wind_gust_speed']} m/s ({compass_direction})")
print(f"Pressure          : {current_weather['pressure']} hPa")
print(f"\nRain Prediction   : {'Yes' if final_rain_prediction else 'No'}")
print(f"(Predicted by best classifier: {best_model_name} — {best_accuracy * 100:.2f}% accuracy)")

print(f"\nFuture Temperature Predictions (via {best_temp_model_name}):")
for time, temp in zip(future_times, future_temp):
    print(f"  {time}  →  {temp:.1f} °C")

print(f"\nFuture Humidity Predictions (via {best_hum_model_name}):")
for time, hum in zip(future_times, future_humidity):
    print(f"  {time}  →  {hum:.1f} %")

print("       MODEL PERFORMANCE SUMMARY        ")

print("Rain Classification Models:")
print(f"  {'Model':<25} {'Accuracy':>10}")
print("  " + "-" * 37)
for name, acc in sorted(accuracy_scores.items(), key=lambda x: x[1], reverse=True):
    marker = " <-- BEST" if name == best_model_name else ""
    print(f"  {name:<25} {acc * 100:>9.2f}%{marker}")
print(f"  {'Voting Ensemble':<25} {voting_acc * 100:>9.2f}%{' <-- BEST' if best_model_name == 'Voting Ensemble' else ''}")

print("\nTemperature Regression Models (lower RMSE = better):")
print(f"  {'Model':<30} {'RMSE':>10}")
print("  " + "-" * 42)
for name, rmse in sorted(temp_mse_scores.items(), key=lambda x: x[1]):
    marker = " <-- BEST" if name == best_temp_model_name else ""
    print(f"  {name:<30} {rmse:>10.4f}{marker}")

print("\nHumidity Regression Models (lower RMSE = better):")
print(f"  {'Model':<30} {'RMSE':>10}")
print("  " + "-" * 42)
for name, rmse in sorted(hum_mse_scores.items(), key=lambda x: x[1]):
    marker = " <-- BEST" if name == best_hum_model_name else ""
    print(f"  {name:<30} {rmse:>10.4f}{marker}")

Enter the city name:  Hyderabad



Fetched current weather for Hyderabad, IN
Historical data loaded: 363 rows, 8 columns

========== RAIN PREDICTION MODEL COMPARISON ==========

--- Random Forest ---
Accuracy: 84.93%
              precision    recall  f1-score   support

     No Rain       0.85      0.98      0.91        57
        Rain       0.86      0.38      0.52        16

    accuracy                           0.85        73
   macro avg       0.85      0.68      0.72        73
weighted avg       0.85      0.85      0.83        73

--- XGBoost ---
Accuracy: 86.30%
              precision    recall  f1-score   support

     No Rain       0.86      0.98      0.92        57
        Rain       0.88      0.44      0.58        16

    accuracy                           0.86        73
   macro avg       0.87      0.71      0.75        73
weighted avg       0.86      0.86      0.84        73

--- Gradient Boosting ---
Accuracy: 84.93%
              precision    recall  f1-score   support

     No Rain       0.85      0.9